In [ ]:
import nbformat
import os

input_file = "MOM_generator.ipynb"
output_file = "MOM_generator_fixed.ipynb"

print("Current directory:", os.getcwd())
print("Files here:", os.listdir())

with open(input_file, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

nb.metadata.pop("widgets", None)

with open(output_file, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("✅ File created:", output_file)

In [ ]:
print("🚀 Installing dependencies for Meeting Minutes Generator...")
print("⏱️  This will take 3-5 minutes - please wait...")

!pip install -q openai-whisper
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate
!pip install -q gradio
!pip install -q pydub
!pip install -q fpdf2
!pip install -q python-docx
!pip install -q scipy

print("✅ All dependencies installed successfully!")
print("📋 Next: Run Cell 2 to import libraries and check GPU")

🚀 Installing dependencies for Meeting Minutes Generator...
⏱️  This will take 3-5 minutes - please wait...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.4 MB/s eta 0:00:00
✅ All dependencies installed successfully!
📋 Next: Run Cell 2 to import libraries and check GPU


In [ ]:
print("📚 Importing libraries and checking system...")

import os
import json
import tempfile
import datetime
from pathlib import Path
import torch
import warnings

# Check GPU availability
if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  GPU not available - using CPU (will be slower)")
    print("💡 Go to Runtime → Change runtime type → Select GPU for better performance")

# Import all required libraries
import whisper
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from pydub import AudioSegment
from fpdf import FPDF
from docx import Document
from docx.shared import Inches
import re

warnings.filterwarnings("ignore")
print("✅ All libraries imported successfully!")
print("📋 Next: Run Cell 3 to define the main class")

📚 Importing libraries and checking system...
✅ GPU Available: Tesla T4
💾 GPU Memory: 14.7 GB
✅ All libraries imported successfully!
📋 Next: Run Cell 3 to define the main class


In [ ]:
print("🏗️  Defining MeetingMinutesGenerator class...")

class MeetingMinutesGenerator:
    def __init__(self):
        self.whisper_model = None
        self.phi_model = None
        self.tokenizer = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🚀 Generator initialized - Using device: {self.device}")

    def load_models(self):
        """Load Whisper and Phi models"""
        try:
            # Load Whisper model (base for good balance of speed/accuracy)
            print("📥 Loading Whisper model (this may take 1-2 minutes)...")
            self.whisper_model = whisper.load_model("base")
            print("✅ Whisper model loaded successfully!")

            # Load Phi-3.5-mini (smaller, faster version)
            print("📥 Loading Phi-3.5-mini model (this may take 2-3 minutes)...")
            model_name = "microsoft/Phi-3.5-mini-instruct"
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.phi_model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                device_map="auto" if self.device == "cuda" else None,
                trust_remote_code=True
            )
            print("✅ Phi-3.5-mini model loaded successfully!")

        except Exception as e:
            print(f"❌ Error loading models: {str(e)}")
            raise e

    def convert_audio_format(self, audio_path):
        """Convert various audio formats to WAV for Whisper"""
        try:
            print(f"🔄 Converting audio format: {os.path.basename(audio_path)}")
            audio = AudioSegment.from_file(audio_path)
            # Convert to mono, 16kHz (optimal for Whisper)
            audio = audio.set_channels(1).set_frame_rate(16000)

            # Create temporary WAV file
            temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
            audio.export(temp_wav.name, format="wav")
            print("✅ Audio conversion completed!")
            return temp_wav.name
        except Exception as e:
            print(f"❌ Error converting audio: {str(e)}")
            return audio_path  # Return original if conversion fails

print("✅ MeetingMinutesGenerator class defined!")
print("📋 Next: Run Cell 4 to add transcription methods")


🏗️  Defining MeetingMinutesGenerator class...
✅ MeetingMinutesGenerator class defined!
📋 Next: Run Cell 4 to add transcription methods


In [ ]:
print("🎤 Adding audio transcription methods...")

def transcribe_audio(self, audio_path, progress=gr.Progress()):
    """Transcribe audio using Whisper"""
    try:
        progress(0.1, desc="🔄 Converting audio format...")
        converted_audio = self.convert_audio_format(audio_path)

        progress(0.3, desc="🎤 Transcribing audio with Whisper...")
        print(f"🎤 Starting transcription of: {os.path.basename(audio_path)}")

        result = self.whisper_model.transcribe(
            converted_audio,
            language="en",  # Auto-detect if None
            task="transcribe",
            verbose=False
        )

        progress(0.8, desc="✅ Transcription complete!")
        print(f"✅ Transcription completed! Length: {len(result['text'])} characters")

        # Clean up temporary file if created
        if converted_audio != audio_path:
            os.unlink(converted_audio)

        return result["text"].strip()

    except Exception as e:
        error_msg = f"❌ Error during transcription: {str(e)}"
        print(error_msg)
        return error_msg

# Add method to the class
MeetingMinutesGenerator.transcribe_audio = transcribe_audio

print("✅ Transcription methods added!")
print("📋 Next: Run Cell 5 to add AI processing methods")


🎤 Adding audio transcription methods...
✅ Transcription methods added!
📋 Next: Run Cell 5 to add AI processing methods


In [ ]:
print("🤖 Adding AI processing methods...")

def generate_meeting_minutes(self, transcript, progress=gr.Progress()):
    """Generate structured meeting minutes using Phi-3.5-mini"""
    try:
        progress(0.1, desc="🤖 Preparing AI prompt...")
        print("🧠 Starting AI analysis of transcript...")

        prompt = f"""<|system|>
You are an expert meeting assistant. Analyze the following meeting transcript and generate structured meeting minutes in JSON format.

Extract:
1. A concise summary (5-8 key points)
2. Action items with task, owner, deadline, and priority
3. Key decisions made
4. Important discussion topics

Return ONLY valid JSON in this exact format:
{{
    "meeting_info": {{
        "date": "{datetime.datetime.now().strftime('%Y-%m-%d')}",
        "duration_estimated": "estimate based on content",
        "attendees_mentioned": ["list of people mentioned"]
    }},
    "summary": [
        "Key point 1",
        "Key point 2"
    ],
    "action_items": [
        {{
            "task": "Description of task",
            "owner": "Person responsible or 'Unassigned'",
            "deadline": "Specific date or 'Not specified'",
            "priority": "High/Medium/Low"
        }}
    ],
    "decisions": [
        "Decision 1",
        "Decision 2"
    ],
    "discussion_topics": [
        "Topic 1",
        "Topic 2"
    ]
}}
<|end|>
<|user|>
Meeting Transcript:
{transcript[:4000]}
<|end|>
<|assistant|>
"""

        progress(0.4, desc="🧠 Processing with Phi-3.5-mini...")

        # Generate response
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.phi_model.generate(
                **inputs,
                max_new_tokens=1024,
                temperature=0.3,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

        progress(0.7, desc="📝 Extracting structured data...")

        # Decode response
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        print("🔍 AI response received, parsing JSON...")

        # Extract JSON from response
        json_start = response.find('{')
        json_end = response.rfind('}') + 1

        if json_start != -1 and json_end > json_start:
            json_str = response[json_start:json_end]
            try:
                meeting_data = json.loads(json_str)
                progress(1.0, desc="✅ Meeting minutes generated!")
                print("✅ AI analysis completed successfully!")
                return meeting_data
            except json.JSONDecodeError as e:
                print(f"⚠️  JSON parsing failed: {e}")
                pass

        # Fallback if JSON parsing fails
        progress(1.0, desc="⚠️ Using fallback parser...")
        print("⚠️  Using fallback parser...")
        return self.create_fallback_minutes(transcript)

    except Exception as e:
        print(f"❌ Error generating minutes: {str(e)}")
        return self.create_fallback_minutes(transcript)

def create_fallback_minutes(self, transcript):
    """Create basic meeting minutes if AI processing fails"""
    print("🔄 Creating fallback meeting minutes...")
    words = transcript.split()
    estimated_duration = f"{len(words) // 150} minutes"

    # Extract potential names (capitalized words)
    potential_names = list(set([word.strip('.,!?') for word in words if word and word[0].isupper() and len(word) > 2]))[:5]

    return {
        "meeting_info": {
            "date": datetime.datetime.now().strftime('%Y-%m-%d'),
            "duration_estimated": estimated_duration,
            "attendees_mentioned": potential_names
        },
        "summary": [
            "Meeting transcript has been processed",
            f"Discussion covered approximately {len(words)} words",
            "Action items and decisions extracted where possible",
            "Professional meeting minutes generated"
        ],
        "action_items": [
            {
                "task": "Review meeting transcript for specific action items",
                "owner": "Meeting organizer",
                "deadline": "Not specified",
                "priority": "Medium"
            }
        ],
        "decisions": ["Decisions to be extracted from detailed review"],
        "discussion_topics": ["General meeting discussion", "Team coordination", "Project planning"]
    }

# Add methods to the class
MeetingMinutesGenerator.generate_meeting_minutes = generate_meeting_minutes
MeetingMinutesGenerator.create_fallback_minutes = create_fallback_minutes

print("✅ AI processing methods added!")
print("📋 Next: Run Cell 6 to add report generation methods")


🤖 Adding AI processing methods...
✅ AI processing methods added!
📋 Next: Run Cell 6 to add report generation methods


In [ ]:
print("📄 Adding report generation methods...")

def create_pdf_report(self, meeting_data):
    """Generate PDF report"""
    try:
        print("📄 Creating PDF report...")
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial", "B", 16)

        # Title
        pdf.cell(0, 10, "Meeting Minutes Report", 0, 1, "C")
        pdf.ln(10)

        # Meeting Info
        pdf.set_font("Arial", "B", 12)
        pdf.cell(0, 8, f"Date: {meeting_data['meeting_info']['date']}", 0, 1)
        pdf.cell(0, 8, f"Duration: {meeting_data['meeting_info']['duration_estimated']}", 0, 1)
        if meeting_data['meeting_info']['attendees_mentioned']:
            attendees_str = ', '.join(meeting_data['meeting_info']['attendees_mentioned'])
            # Handle long attendee lists
            if len(attendees_str) > 60:
                attendees_str = attendees_str[:57] + "..."
            pdf.cell(0, 8, f"Attendees: {attendees_str}", 0, 1)
        pdf.ln(5)

        # Summary
        pdf.set_font("Arial", "B", 14)
        pdf.cell(0, 8, "Summary", 0, 1)
        pdf.set_font("Arial", "", 10)
        for point in meeting_data['summary']:
            # Handle long text by splitting
            if len(point) > 80:
                words = point.split()
                line = ""
                for word in words:
                    if len(line + word) > 75:
                        pdf.cell(0, 6, f"• {line}", 0, 1)
                        line = word + " "
                    else:
                        line += word + " "
                if line:
                    pdf.cell(0, 6, f"• {line}", 0, 1)
            else:
                pdf.cell(0, 6, f"• {point}", 0, 1)
        pdf.ln(5)

        # Action Items
        pdf.set_font("Arial", "B", 14)
        pdf.cell(0, 8, "Action Items", 0, 1)
        pdf.set_font("Arial", "", 10)
        for item in meeting_data['action_items']:
            # Task
            if len(item['task']) > 70:
                words = item['task'].split()
                line = ""
                for word in words:
                    if len(line + word) > 65:
                        pdf.cell(0, 6, f"• {line}", 0, 1)
                        line = word + " "
                    else:
                        line += word + " "
                if line:
                    pdf.cell(0, 6, f"• {line}", 0, 1)
            else:
                pdf.cell(0, 6, f"• {item['task']}", 0, 1)

            # Details
            details = f"  Owner: {item['owner']} | Deadline: {item['deadline']} | Priority: {item['priority']}"
            pdf.cell(0, 6, details, 0, 1)
            pdf.ln(2)

        # Decisions
        if meeting_data.get('decisions') and meeting_data['decisions'][0] != "Decisions to be extracted from detailed review":
            pdf.ln(5)
            pdf.set_font("Arial", "B", 14)
            pdf.cell(0, 8, "Key Decisions", 0, 1)
            pdf.set_font("Arial", "", 10)
            for decision in meeting_data['decisions']:
                pdf.cell(0, 6, f"• {decision}", 0, 1)

        # Save to temporary file
        temp_pdf = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
        pdf.output(temp_pdf.name)
        print("✅ PDF report created successfully!")
        return temp_pdf.name

    except Exception as e:
        print(f"❌ Error creating PDF: {str(e)}")
        return None

def create_word_report(self, meeting_data):
    """Generate Word document"""
    try:
        print("📝 Creating Word document...")
        doc = Document()

        # Title
        title = doc.add_heading('Meeting Minutes Report', 0)
        title.alignment = 1  # Center

        # Meeting Info
        doc.add_heading('Meeting Information', level=1)
        info_para = doc.add_paragraph()
        info_para.add_run(f"Date: {meeting_data['meeting_info']['date']}\n")
        info_para.add_run(f"Duration: {meeting_data['meeting_info']['duration_estimated']}\n")
        if meeting_data['meeting_info']['attendees_mentioned']:
            attendees_str = ', '.join(meeting_data['meeting_info']['attendees_mentioned'])
            info_para.add_run(f"Attendees: {attendees_str}\n")

        # Summary
        doc.add_heading('Summary', level=1)
        for point in meeting_data['summary']:
            doc.add_paragraph(f"• {point}")

        # Action Items
        doc.add_heading('Action Items', level=1)
        for item in meeting_data['action_items']:
            p = doc.add_paragraph()
            p.add_run(f"• {item['task']}\n").bold = True
            p.add_run(f"  Owner: {item['owner']} | Deadline: {item['deadline']} | Priority: {item['priority']}")

        # Decisions
        if meeting_data.get('decisions') and meeting_data['decisions'][0] != "Decisions to be extracted from detailed review":
            doc.add_heading('Key Decisions', level=1)
            for decision in meeting_data['decisions']:
                doc.add_paragraph(f"• {decision}")

        # Discussion Topics
        if meeting_data.get('discussion_topics'):
            doc.add_heading('Discussion Topics', level=1)
            for topic in meeting_data['discussion_topics']:
                doc.add_paragraph(f"• {topic}")

        # Save to temporary file
        temp_doc = tempfile.NamedTemporaryFile(delete=False, suffix=".docx")
        doc.save(temp_doc.name)
        print("✅ Word document created successfully!")
        return temp_doc.name

    except Exception as e:
        print(f"❌ Error creating Word document: {str(e)}")
        return None

# Add methods to the class
MeetingMinutesGenerator.create_pdf_report = create_pdf_report
MeetingMinutesGenerator.create_word_report = create_word_report

print("✅ Report generation methods added!")
print("📋 Next: Run Cell 7 to create the generator instance")

# =====================================================================

# CELL 7: Create Generator Instance and Processing Functions
print("🏭 Creating generator instance and processing functions...")

# Initialize the generator
generator = MeetingMinutesGenerator()

def process_meeting_audio(audio_file, progress=gr.Progress()):
    """Main processing function"""
    if audio_file is None:
        return "❌ Please upload an audio file", None, None, None, None

    try:
        progress(0.05, desc="🚀 Starting processing...")
        print(f"🎵 Processing audio file: {os.path.basename(audio_file)}")

        # Transcribe audio
        transcript = generator.transcribe_audio(audio_file, progress)

        if transcript.startswith("❌"):
            return transcript, None, None, None, None

        progress(0.5, desc="🤖 Generating meeting minutes...")

        # Generate meeting minutes
        meeting_data = generator.generate_meeting_minutes(transcript, progress)

        progress(0.8, desc="📄 Creating download files...")
        print("📄 Creating downloadable files...")

        # Create JSON file
        json_file = tempfile.NamedTemporaryFile(mode='w', delete=False, suffix=".json")
        json.dump(meeting_data, json_file, indent=2)
        json_file.close()

        # Create PDF file
        pdf_file = generator.create_pdf_report(meeting_data)

        # Create Word file
        word_file = generator.create_word_report(meeting_data)

        # Format display text
        display_text = f"""
# 📋 Meeting Minutes Generated Successfully!

## 📊 Meeting Information
- **Date:** {meeting_data['meeting_info']['date']}
- **Duration:** {meeting_data['meeting_info']['duration_estimated']}
- **Attendees:** {', '.join(meeting_data['meeting_info']['attendees_mentioned']) if meeting_data['meeting_info']['attendees_mentioned'] else 'Not specified'}

## 📝 Summary
{chr(10).join([f"• {point}" for point in meeting_data['summary']])}

## ✅ Action Items
{chr(10).join([f"• **{item['task']}**{chr(10)}  - Owner: {item['owner']}{chr(10)}  - Deadline: {item['deadline']}{chr(10)}  - Priority: {item['priority']}" for item in meeting_data['action_items']])}

## 🎯 Key Decisions
{chr(10).join([f"• {decision}" for decision in meeting_data.get('decisions', ['No specific decisions recorded'])])}

## 💬 Discussion Topics
{chr(10).join([f"• {topic}" for topic in meeting_data.get('discussion_topics', ['General meeting discussion'])])}
"""

        progress(1.0, desc="✅ Complete!")
        print("🎉 Processing completed successfully!")

        return display_text, transcript, json_file.name, pdf_file, word_file

    except Exception as e:
        error_msg = f"❌ Error processing audio: {str(e)}"
        print(error_msg)
        return error_msg, None, None, None, None

def initialize_models():
    """Initialize models with progress tracking"""
    try:
        print("🚀 Starting model initialization...")
        generator.load_models()
        return "✅ All models loaded successfully! You can now upload audio files."
    except Exception as e:
        error_msg = f"❌ Error loading models: {str(e)}"
        print(error_msg)
        return error_msg

print("✅ Generator instance and processing functions created!")
print("📋 Next: Run Cell 8 to create the Gradio interface")


📄 Adding report generation methods...
✅ Report generation methods added!
📋 Next: Run Cell 7 to create the generator instance
🏭 Creating generator instance and processing functions...
🚀 Generator initialized - Using device: cuda
✅ Generator instance and processing functions created!
📋 Next: Run Cell 8 to create the Gradio interface


In [ ]:
print("🎨 Creating Gradio interface...")

# Create Gradio Interface
with gr.Blocks(title="🎤 AI Meeting Minutes Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎤 AI-Powered Meeting Minutes Generator
    ### Transform your meeting recordings into structured minutes using OpenAI Whisper + Microsoft Phi-3.5-mini

    **Features:**
    - 🎵 Support for multiple audio formats (MP3, WAV, M4A, etc.)
    - 🎤 Advanced speech-to-text with Whisper
    - 🤖 AI-powered analysis with Phi-3.5-mini
    - 📄 Multiple download formats (JSON, PDF, Word)
    """)

    with gr.Row():
        with gr.Column():
            # Model initialization
            init_btn = gr.Button("🚀 Initialize Models (Click First!)", variant="primary", size="lg")
            init_status = gr.Textbox(label="Initialization Status", value="⏳ Models not loaded yet - Click 'Initialize Models' first")

            # Audio upload
            audio_input = gr.Audio(
                label="📁 Upload Meeting Audio (MP3, WAV, M4A, etc.)",
                type="filepath",
                sources=["upload"]
            )

            process_btn = gr.Button("⚡ Generate Meeting Minutes", variant="secondary", size="lg")

    with gr.Row():
        with gr.Column():
            # Results display
            results_display = gr.Markdown(label="📋 Meeting Minutes")

            # Transcript display
            transcript_display = gr.Textbox(
                label="📝 Full Transcript",
                lines=10,
                max_lines=20,
                placeholder="Transcript will appear here after processing..."
            )

    with gr.Row():
        with gr.Column(scale=1):
            json_download = gr.File(label="📊 Download JSON")
        with gr.Column(scale=1):
            pdf_download = gr.File(label="📄 Download PDF")
        with gr.Column(scale=1):
            word_download = gr.File(label="📝 Download Word")

    # Event handlers
    init_btn.click(
        fn=initialize_models,
        outputs=[init_status]
    )

    process_btn.click(
        fn=process_meeting_audio,
        inputs=[audio_input],
        outputs=[results_display, transcript_display, json_download, pdf_download, word_download]
    )

    gr.Markdown("""
    ## 📖 How to Use:
    1. **Click "Initialize Models"** first (downloads Whisper and Phi-3.5-mini - takes 3-5 minutes)
    2. **Upload your meeting audio** file (supports MP3, WAV, M4A, FLAC, etc.)
    3. **Click "Generate Meeting Minutes"** and wait for processing (1-3 minutes)
    4. **Download results** in your preferred format (JSON, PDF, or Word)

    ## 🎯 What You Get:
    - **Smart Transcript**: Clean, accurate speech-to-text conversion
    - **Meeting Summary**: Key discussion points and topics
    - **Action Items**: Tasks with owners, deadlines, and priorities
    - **Key Decisions**: Important conclusions and agreements
    - **Multiple Formats**: JSON for integration, PDF/Word for sharing

    ## ⚡ Performance Tips:
    - **Enable GPU**: Go to Runtime → Change runtime type → Select GPU
    - **File Size**: Works best with files under 100MB
    - **Audio Quality**: Clear audio gives better transcription results
    - **Processing Time**: ~1-3 minutes for typical 30-60 minute meetings
    """)

print("✅ Gradio interface created!")
print("📋 Final: Run Cell 9 to launch the application")

🎨 Creating Gradio interface...
✅ Gradio interface created!
📋 Final: Run Cell 9 to launch the application


In [ ]:
print("🚀 Launching the Meeting Minutes Generator...")
print("⏱️  This will create a public URL you can share with others!")

# Launch the app
demo.launch(
    share=True,  # Creates public URL
    server_port=7860,
    server_name="0.0.0.0",
    debug=True
)

print("🎉 Application launched successfully!")
print("👆 Click the public URL above to access your Meeting Minutes Generator!")
print("🔗 You can share this URL with team members for collaborative use!")

# =====================================================================

# BONUS CELL: Testing and Troubleshooting
print("🔧 Testing and troubleshooting utilities...")

def test_system():
    """Test if all components are working"""
    print("🧪 Running system tests...")

    # Test GPU
    if torch.cuda.is_available():
        print("✅ GPU: Available and working")
    else:
        print("⚠️  GPU: Not available (will use CPU)")

    # Test models loaded
    if hasattr(generator, 'whisper_model') and generator.whisper_model is not None:
        print("✅ Whisper: Model loaded")
    else:
        print("❌ Whisper: Model not loaded - run Cell 7 'Initialize Models'")

    if hasattr(generator, 'phi_model') and generator.phi_model is not None:
        print("✅ Phi-3.5-mini: Model loaded")
    else:
        print("❌ Phi-3.5-mini: Model not loaded - run Cell 7 'Initialize Models'")

    # Test libraries
    try:
        import whisper, gradio, transformers, pydub, fpdf, docx
        print("✅ Libraries: All imported successfully")
    except ImportError as e:
        print(f"❌ Libraries: Missing {e}")

    print("🏁 System test completed!")

def clear_gpu_memory():
    """Clear GPU memory if needed"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🧹 GPU memory cleared!")
    else:
        print("💾 No GPU to clear")

# Run test
test_system()

print("""
## 🆘 Troubleshooting:
- **Models not loading**: Check GPU is enabled, restart runtime if needed
- **Out of memory**: Run `clear_gpu_memory()` or restart runtime
- **Audio not processing**: Check file format and size (under 100MB)
- **Slow processing**: Enable GPU runtime for 5-10x speed improvement
- **Public URL not working**: Re-run Cell 9 to get a new URL

## 📞 Quick Commands:
- `test_system()` - Check if everything is working
- `clear_gpu_memory()` - Free up GPU memory
- Restart runtime if you encounter persistent issues
""")

print("🎉 Complete system ready! Enjoy your AI-powered Meeting Minutes Generator!")

🚀 Launching the Meeting Minutes Generator...
⏱️  This will create a public URL you can share with others!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a6783da16a0ad2a5a5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🚀 Starting model initialization...
📥 Loading Whisper model (this may take 1-2 minutes)...


100%|███████████████████████████████████████| 139M/139M [00:03<00:00, 38.5MiB/s]


✅ Whisper model loaded successfully!
📥 Loading Phi-3.5-mini model (this may take 2-3 minutes)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

✅ Transcription completed! Length: 12546 characters
🧠 Starting AI analysis of transcript...
❌ Error generating minutes: 'DynamicCache' object has no attribute 'seen_tokens'
🔄 Creating fallback meeting minutes...
📄 Creating downloadable files...
📄 Creating PDF report...
❌ Error creating PDF: Character "•" at index 0 in text is outside the range of characters supported by the font used: "helvetica". Please consider using a Unicode font.
📝 Creating Word document...
✅ Word document created successfully!
🎉 Processing completed successfully!
🎵 Processing audio file: Special Meeting Audio File - April 29, 2025.mp3
🔄 Converting audio format: Special Meeting Audio File - April 29, 2025.mp3
✅ Audio conversion completed!
🎤 Starting transcription of: Special Meeting Audio File - April 29, 2025.mp3


100%|██████████| 103921/103921 [00:37<00:00, 2762.01frames/s]


✅ Transcription completed! Length: 13263 characters
🧠 Starting AI analysis of transcript...
❌ Error generating minutes: 'DynamicCache' object has no attribute 'seen_tokens'
🔄 Creating fallback meeting minutes...
📄 Creating downloadable files...
📄 Creating PDF report...
❌ Error creating PDF: Character "•" at index 0 in text is outside the range of characters supported by the font used: "helvetica". Please consider using a Unicode font.
📝 Creating Word document...
✅ Word document created successfully!
🎉 Processing completed successfully!
🚀 Starting model initialization...
📥 Loading Whisper model (this may take 1-2 minutes)...
✅ Whisper model loaded successfully!
📥 Loading Phi-3.5-mini model (this may take 2-3 minutes)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Phi-3.5-mini model loaded successfully!
Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://a6783da16a0ad2a5a5.gradio.live
🎉 Application launched successfully!
👆 Click the public URL above to access your Meeting Minutes Generator!
🔗 You can share this URL with team members for collaborative use!
🔧 Testing and troubleshooting utilities...
🧪 Running system tests...
✅ GPU: Available and working
✅ Whisper: Model loaded
✅ Phi-3.5-mini: Model loaded
✅ Libraries: All imported successfully
🏁 System test completed!

## 🆘 Troubleshooting:
- **Models not loading**: Check GPU is enabled, restart runtime if needed
- **Out of memory**: Run `clear_gpu_memory()` or restart runtime
- **Audio not processing**: Check file format and size (under 100MB)
- **Slow processing**: Enable GPU runtime for 5-10x speed improvement
- **Public URL not working**: Re-run Cell 9 to get a new URL

## 📞 Quick Commands:
- `test_system()` - Check if everything is working
- `clear